# Campus Lost and Found Portal
**Assignment No 3 - End-to-End Portal Design (Backend)**

**Team:** Ali Abdullah (25L-5091) and Saad Jamil (25L-5032)

A console portal where students and staff report lost or found items, search the reports and generate a summary. It uses functions (`def`), loops (`for`, `while`), CSV file handling and exception handling (`try`, `except`, `finally`).

## How the code is organised (all inside the single code cell below)
1. **Setup:** imports, file names and automatic creation of `users.csv` (default accounts) if it is missing.
2. **`user_login()`:** checks the username and password against `users.csv`.
3. **`add_record()`:** validates the input (type, empty fields, date format, numeric contact) and appends the record to `records.csv`; `finally` always prints a completion message.
4. **`search_record()`:** `for` loop that finds records by keyword.
5. **`generate_report()`:** `for` loop that counts lost, found and per-category records.
6. **Startup:** existing records are read from `records.csv`.
7. **Main program:** login loop (3 attempts), then the `while` menu loop until **Exit**.

**Login accounts:** `ali / ali123`, `saad / saad123`, `admin / admin123` (the password is hidden while typing).

In [ ]:
"""
Campus Lost and Found Portal (backend only)
Assignment 3 - End-to-End Portal Design
Team: Ali Abdullah (25L-5091) and Saad Jamil (25L-5032)
"""

import csv
from datetime import datetime
from getpass import getpass

USERS_FILE = "users.csv"
RECORDS_FILE = "records.csv"
FIELDS = ["record_id", "type", "item_name", "category", "location", "date", "contact", "reported_by"]

# Create the login file with default accounts if it does not exist yet (needed in Google Colab)
try:
    open(USERS_FILE, "r").close()
except FileNotFoundError:
    try:
        with open(USERS_FILE, "w") as file:
            file.write("username,password,role\n")
            file.write("admin,admin123,staff\n")
            file.write("ali,ali123,student\n")
            file.write("saad,saad123,student\n")
        print("users.csv created with default accounts.")
    except PermissionError:
        print("Error: cannot create the users file.")


def user_login(username, password):
    """Check a username and password against users.csv.

    Parameters:
        username (str): name typed by the user.
        password (str): password typed by the user.
    Returns:
        str: the username if the login is correct, otherwise None.
    """
    try:
        with open(USERS_FILE, "r", newline="") as file:
            for row in csv.DictReader(file):
                if row["username"] == username and row["password"] == password:
                    return username
    except FileNotFoundError:
        print("Error: users file not found.")
    except PermissionError:
        print("Error: no permission to read the users file.")
    return None


def add_record(records, item_type, item_name, category, location, date_text, contact, reported_by):
    """Validate a new lost/found report, add it to the list and append it to records.csv.

    Parameters:
        records (list): all records currently loaded in memory.
        item_type (str): 'lost' or 'found'.
        item_name, category, location, contact (str): report details.
        date_text (str): date in YYYY-MM-DD format.
        reported_by (str): logged in username.
    Returns:
        dict: the saved record, or None if the input was invalid or saving failed.
    """
    saved = None
    try:
        item_type = item_type.strip().lower()
        if item_type not in ("lost", "found"):
            raise ValueError("type must be 'lost' or 'found'")
        if item_name.strip() == "" or location.strip() == "":
            raise ValueError("item name and location cannot be empty")
        datetime.strptime(date_text, "%Y-%m-%d")      # raises ValueError for a bad date
        if not contact.strip().isdigit():
            raise ValueError("contact must be a numeric phone number")

        new_record = {
            "record_id": len(records) + 1,
            "type": item_type,
            "item_name": item_name.strip(),
            "category": category.strip().lower(),
            "location": location.strip(),
            "date": date_text,
            "contact": contact.strip(),
            "reported_by": reported_by,
        }
        with open(RECORDS_FILE, "a", newline="") as file:
            csv.DictWriter(file, fieldnames=FIELDS).writerow(new_record)
        records.append(new_record)
        saved = new_record
    except ValueError as error:
        print("Invalid input:", error)
    except PermissionError:
        print("Error: no permission to write to the records file.")
    except FileNotFoundError:
        print("Error: records file path not found.")
    finally:
        print("Add-record operation finished.")
    return saved


def search_record(records, keyword):
    """Find records whose item name, category or location contains the keyword.

    Parameters:
        records (list): all records loaded in memory.
        keyword (str): text to look for (not case sensitive).
    Returns:
        list: matching records (empty list if nothing matches).
    """
    matches = []
    keyword = keyword.strip().lower()
    for record in records:
        searchable = (record["item_name"] + " " + record["category"] + " " + record["location"]).lower()
        if keyword != "" and keyword in searchable:
            matches.append(record)
    return matches


def generate_report(records):
    """Count the stored records by type and by category.

    Parameters:
        records (list): all records loaded in memory.
    Returns:
        dict: totals for lost/found items and a per-category count.
    """
    report = {"total": 0, "lost": 0, "found": 0, "by_category": {}}
    for record in records:
        report["total"] += 1
        report[record["type"]] += 1
        category = record["category"]
        report["by_category"][category] = report["by_category"].get(category, 0) + 1
    return report


# ---------------------------------------------------------------- main program
records = []
try:                                                    # read existing records on startup
    with open(RECORDS_FILE, "r", newline="") as file:
        for row in csv.DictReader(file):
            records.append(row)
    print("Loaded", len(records), "existing records.")
except FileNotFoundError:
    print("No records file found - creating a new one.")
    try:
        with open(RECORDS_FILE, "w", newline="") as file:
            csv.DictWriter(file, fieldnames=FIELDS).writeheader()
    except PermissionError:
        print("Error: cannot create the records file.")
except PermissionError:
    print("Error: no permission to read the records file.")

print("=== Campus Lost and Found Portal ===")
current_user = None
attempts = 0
while current_user is None and attempts < 3:           # login loop
    name = input("Username: ")
    pwd = getpass("Password: ")   # typed characters are hidden
    current_user = user_login(name, pwd)
    if current_user is None:
        attempts += 1
        print("Wrong username or password.")

while current_user is not None:                        # main menu loop
    print("\n1. Add record   2. Search records   3. Generate report   4. Exit")
    choice = input("Choose an option: ")
    if choice == "1":
        add_record(records,
                   input("Type (lost/found): "), input("Item name: "), input("Category: "),
                   input("Location: "), input("Date (YYYY-MM-DD): "), input("Contact number: "),
                   current_user)
    elif choice == "2":
        results = search_record(records, input("Search keyword: "))
        if len(results) == 0:
            print("No matching records.")
        for record in results:
            print(record["record_id"], record["type"].upper(), record["item_name"], "|",
                  record["location"], "|", record["date"], "|", record["contact"])
    elif choice == "3":
        summary = generate_report(records)
        print("Total:", summary["total"], "| Lost:", summary["lost"], "| Found:", summary["found"])
        for category, count in summary["by_category"].items():
            print(" -", category, ":", count)
    elif choice == "4":
        print("Goodbye,", current_user)
        break
    else:
        print("Invalid choice, try again.")

users.csv created with default accounts.
No records file found - creating a new one.
=== Campus Lost and Found Portal ===
Username: admin
Password: ·····
Wrong username or password.
Username: ali
Password: ······

1. Add record   2. Search records   3. Generate report   4. Exit
Choose an option: 1
Type (lost/found): lost
Item name: Blue Water Bottle
Category: accessories
Location: Library 2nd Floor
Date (YYYY-MM-DD): 2026-11-05
Contact number: 03001234567
Add-record operation finished.

1. Add record   2. Search records   3. Generate report   4. Exit
Choose an option: 1
Type (lost/found): found
Item name: Black Wallet
Category: wallet
Location: Cafeteria
Date (YYYY-MM-DD): 2026-11-06
Contact number: 03211234567
Add-record operation finished.

1. Add record   2. Search records   3. Generate report   4. Exit
Choose an option: 1
Type (lost/found): found
Item name: Casio Calculator
Category: electronics
Location: CS Lab 3
Date (YYYY-MM-DD): 06-11-2026
Contact number: 03111234567
Invalid in

## Notes on the saved output
The output above is a sample run with scripted answers. It shows a failed login, a successful login, valid records being added, an **invalid date** and an **invalid phone number** being rejected, a search with and without results, the report, an invalid menu choice and exit. Run the cell yourself to type your own input.